# Hospital Operations & Patient Analytics

### Project Overview
This project focuses on analyzing and cleaning hospital patient data using Python and Pandas.

**Objectives:**
- Understand the structure and quality of the dataset
- Identify and handle missing and inconsistent values
- Detect invalid records and duplicates
- Perform data transformation and feature engineering
- Export the final cleaned dataset

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

## Load Dataset

In [2]:
df = pd.read_csv(r"C:\Users\Shakshi\OneDrive\Desktop\Hospital_Analytics\hospital_patient_analytics_expanded.csv")
df.head()

,Patient_ID,Patient_Name,Age,Gender,City,Department,Doctor_Name,Admission_Date,Discharge_Date,Insurance_Provider,Bill_Amount_INR,Patient_Satisfaction_Score
0,PT011500,Anvi Mahal,44.0,female,Hyderabad,ENT,Dr. Manoj Tiwari,"Aug 06, 2021",2021-08-07,ICICI Lombard,51949.68,4.0
1,PT006476,Janya Hayer,72.0,f,Bhopal,Cardiology,Dr. Sunita Rao,"Apr 03, 2021",2021-04-17,Star Health,133100.60,3.0
2,PT013168,Jagdish Nagi,20.0,MALE,Ahmedabad,Neurology,Dr. Priya Nair,24 July 2024,2024-07-30,Max Bupa,213515.14,4.0
3,PT000863,Ekta Luthra,59.0,F,Kolkata,Orthopedics,Dr. Neha Verma,2021-08-28,2021-08-29,Max Bupa,139055.83,3.0
4,PT005971,Ikbal Dar,58.0,M,Chennai,Dermatology,Dr. Shalini Agarwal,2021-10-15,2021-10-18,Religare Health,42811.13,2.0


## Exploratory Data Analysis

### Gender Data Quality Check

The Gender column contains inconsistent representations such as `M`, `m`, `Male`, `F`, `f`, and `Female`.

In [3]:
df["Gender"].value_counts(dropna=False)

Gender
male      1511
female    1479
M         1471
F         1465
FEMALE    1455
MALE      1443
Male      1428
m         1424
Female    1381
f         1364
other      199
Other      196
O          184
Name: count, dtype: int64

### Data Cleaning: Gender

The inconsistent gender values are standardized into `Male` and `Female` categories.

In [4]:
df["Gender"] = (
    df["Gender"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "m": "Male",
        "male": "Male",
        "f": "Female",
        "female": "Female"
    })
)

In [5]:
df["Gender"].value_counts()

Gender
Male      7277
Female    7144
other      395
o          184
Name: count, dtype: int64

### Insight

Gender values have been standardized into consistent categories for analysis.

In [6]:
df.shape

(15000, 12)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Patient_ID                  15000 non-null  object 
 1   Patient_Name                15000 non-null  object 
 2   Age                         14550 non-null  float64
 3   Gender                      15000 non-null  object 
 4   City                        14700 non-null  object 
 5   Department                  15000 non-null  object 
 6   Doctor_Name                 15000 non-null  object 
 7   Admission_Date              15000 non-null  object 
 8   Discharge_Date              14625 non-null  object 
 9   Insurance_Provider          13122 non-null  object 
 10  Bill_Amount_INR             15000 non-null  float64
 11  Patient_Satisfaction_Score  13998 non-null  float64
dtypes: float64(3), object(9)
memory usage: 1.4+ MB


### Insight
The dataset contains 15,000 records and 12 columns. Several columns contain missing values, particularly Insurance_Provider and Patient_Satisfaction_Score, which require further data-quality treatment.

In [8]:
df.describe(include="all")

,Patient_ID,Patient_Name,Age,Gender,City,Department,Doctor_Name,Admission_Date,Discharge_Date,Insurance_Provider,Bill_Amount_INR,Patient_Satisfaction_Score
count,15000,15000,14550.000000,15000,14700,15000,15000,15000,14625,13122,15000.000000,13998.000000
unique,15000,14556,NaN,4,16,12,27,6302,6206,9,NaN,NaN
top,PT011500,Udyati Patla,NaN,Male,Nagpur,General Medicine,Dr. Amit Deshmukh,2023-04-23,2024-05-15,Star Health,NaN,NaN
freq,1,3,NaN,7277,964,1300,675,14,12,2223,NaN,NaN
mean,NaN,NaN,43.511203,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100195.287127,3.757465
std,NaN,NaN,19.631938,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81444.253403,0.884951
min,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-19881.290000,1.000000
25%,NaN,NaN,31.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39343.830000,3.000000
50%,NaN,NaN,45.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75416.100000,4.000000
75%,NaN,NaN,58.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,140881.082500,4.000000


## Missing Value Analysis

In [9]:
missing_count = df.isnull().sum()

missing_percentage = (missing_count / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Values": missing_count,
    "Missing %": missing_percentage.round(2)
})

missing_df[missing_df["Missing Values"] > 0]

,Missing Values,Missing %
Age,450,3.00
City,300,2.00
Discharge_Date,375,2.50
Insurance_Provider,1878,12.52
Patient_Satisfaction_Score,1002,6.68


In [10]:
missing_df.sort_values("Missing %", ascending=False)

,Missing Values,Missing %
Insurance_Provider,1878,12.52
Patient_Satisfaction_Score,1002,6.68
Age,450,3.00
Discharge_Date,375,2.50
City,300,2.00
Gender,0,0.00
Patient_ID,0,0.00
Patient_Name,0,0.00
Admission_Date,0,0.00
Doctor_Name,0,0.00


### Insight
Missing values were identified across five columns. Insurance_Provider has the highest missing percentage at 12.52%, followed by Patient_Satisfaction_Score at 6.68%.

In [11]:
missing_df[missing_df["Missing %"] > 2]

,Missing Values,Missing %
Age,450,3.00
Discharge_Date,375,2.50
Insurance_Provider,1878,12.52
Patient_Satisfaction_Score,1002,6.68


## Data Cleaning

In [12]:
# Remove records with negative bill amounts
df = df[df["Bill_Amount_INR"] >= 0].copy()

In [13]:
df["Bill_Amount_INR"].describe()

count     14889.000000
mean     101019.614789
std       81182.180174
min        4371.460000
25%       40081.150000
50%       76129.970000
75%      141419.150000
max      610252.070000
Name: Bill_Amount_INR, dtype: float64

In [14]:
# Remove duplicate patient records based on Patient_ID
df = df.drop_duplicates(subset="Patient_ID")

## Data Transformation

The dataset is transformed by converting date columns into datetime format and handling missing values using appropriate statistical or categorical replacement methods.

In [15]:
df["Admission_Date"] = pd.to_datetime(
    df["Admission_Date"],
    format="mixed"
)

df["Discharge_Date"] = pd.to_datetime(
    df["Discharge_Date"],
    format="mixed"
)

In [16]:
df["Age"] = df["Age"].fillna(df["Age"].median())

df["City"] = df["City"].fillna(df["City"].mode()[0])

df["Insurance_Provider"] = df["Insurance_Provider"].fillna("None")

df["Patient_Satisfaction_Score"] = df["Patient_Satisfaction_Score"].fillna(
    df["Patient_Satisfaction_Score"].median()
)

df["Discharge_Date"] = df["Discharge_Date"].fillna(df["Admission_Date"])

In [17]:
df["Department"].value_counts()

Department
General Medicine    1284
Urology             1275
Pediatrics          1262
Gastroenterology    1259
Neurology           1256
Orthopedics         1244
Oncology            1233
Dermatology         1226
Nephrology          1220
Gynecology          1218
Cardiology          1210
ENT                 1202
Name: count, dtype: int64

### Insight

General Medicine has the highest number of patient records, while ENT has the lowest among the departments.

## Feature Engineering
New analytical features are created to support patient-level analysis, including length of stay, admission year, month, quarter, age group, bill category, admission day, and stay category.

In [18]:
df["Length_of_Stay"] = (
    df["Discharge_Date"]-
    df["Admission_Date"]
).dt.days

df = df[df["Length_of_Stay"]>=0]

df["Admission_Year"] = df["Admission_Date"].dt.year

df["Admission_Month"] = df["Admission_Date"].dt.month_name()

df["Admission_Quarter"] = df["Admission_Date"].dt.quarter

In [20]:
# Age Group
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[-float("inf"), 18, 35, 50, 65, float("inf")],
    labels=["0-18", "19-35", "36-50", "51-65", "65+"]
)

# Bill Category
df["Bill_Category"] = pd.cut(
    df["Bill_Amount_INR"],
    bins=[-float("inf"), 10000, 30000, 60000, float("inf")],
    labels=["Low", "Medium", "High", "Very High"]
)

# Admission Day
df["Admission_Day"] = df["Admission_Date"].dt.day_name()

# Stay Category
df["Stay_Category"] = pd.cut(
    df["Length_of_Stay"],
    bins=[-float("inf"), 0, 3, 7, float("inf")],
    labels=["Same Day", "1-3 Days", "4-7 Days", "More than 7 Days"]
)

In [26]:
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())

Patient_ID                    0
Patient_Name                  0
Age                           0
Gender                        0
City                          0
Department                    0
Doctor_Name                   0
Admission_Date                0
Discharge_Date                0
Insurance_Provider            0
Bill_Amount_INR               0
Patient_Satisfaction_Score    0
Length_of_Stay                0
Admission_Year                0
Admission_Month               0
Admission_Quarter             0
Age_Group                     0
Bill_Category                 0
Admission_Day                 0
Stay_Category                 0
dtype: int64

Total missing values: 0


In [21]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 14734 entries, 0 to 14999
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Patient_ID                  14734 non-null  object        
 1   Patient_Name                14734 non-null  object        
 2   Age                         14734 non-null  float64       
 3   Gender                      14734 non-null  object        
 4   City                        14734 non-null  object        
 5   Department                  14734 non-null  object        
 6   Doctor_Name                 14734 non-null  object        
 7   Admission_Date              14734 non-null  datetime64[ns]
 8   Discharge_Date              14734 non-null  datetime64[ns]
 9   Insurance_Provider          14734 non-null  object        
 10  Bill_Amount_INR             14734 non-null  float64       
 11  Patient_Satisfaction_Score  14734 non-null  float64       


In [22]:
df.isnull().sum()

Patient_ID                    0
Patient_Name                  0
Age                           0
Gender                        0
City                          0
Department                    0
Doctor_Name                   0
Admission_Date                0
Discharge_Date                0
Insurance_Provider            0
Bill_Amount_INR               0
Patient_Satisfaction_Score    0
Length_of_Stay                0
Admission_Year                0
Admission_Month               0
Admission_Quarter             0
Age_Group                     0
Bill_Category                 0
Admission_Day                 0
Stay_Category                 0
dtype: int64

In [23]:
df.describe()

,Age,Admission_Date,Discharge_Date,Bill_Amount_INR,Patient_Satisfaction_Score,Length_of_Stay,Admission_Year,Admission_Quarter
count,14734.000000,14734,14734,14734.000000,14734.000000,14734.000000,14734.000000,14734.000000
mean,43.556197,2023-01-08 09:25:34.898873600,2023-01-11 19:06:01.123930880,101016.437523,3.771820,3.403081,2022.524365,2.493960
min,0.000000,2020-01-01 00:00:00,2020-01-01 00:00:00,4371.460000,1.000000,0.000000,2020.000000,1.000000
25%,31.000000,2021-07-13 00:00:00,2021-07-16 00:00:00,40071.170000,3.000000,0.000000,2021.000000,1.000000
50%,45.000000,2023-01-25 00:00:00,2023-01-28 00:00:00,76093.095000,4.000000,2.000000,2023.000000,3.000000
75%,57.000000,2024-07-06 00:00:00,2024-07-09 18:00:00,141419.345000,4.000000,5.000000,2024.000000,3.000000
max,95.000000,2025-12-31 00:00:00,2026-01-14 00:00:00,610252.070000,5.000000,45.000000,2025.000000,4.000000
std,19.337042,NaN,NaN,81255.312195,0.859697,4.374956,1.704434,1.114042


### Insight

After cleaning and transformation, the dataset contains 14,734 records and 20 analytical columns with no remaining missing values.

## Export Cleaned Dataset

In [24]:
df.to_csv(
    "hospital_operations_patient_analytics_cleaned.csv",
    index=False
)

In [25]:
print("Data cleaning completed successfully.")
print(df.shape)

Data cleaning completed successfully.
(14734, 20)


## Conclusion

The hospital patient dataset was successfully cleaned and transformed using Python and Pandas. Missing values, duplicate records, inconsistent categorical values, and invalid billing records were addressed. Additional features were created to support further hospital operations and patient analytics.

The final cleaned dataset was exported for further analysis and visualization.